## 1

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # Use GPU 1 which has 40GB free
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Reduce fragmentation

# Then your existing imports and code...

import os
import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import RobertaTokenizer  # Kept solely for its character/subword vocabulary mapping
import timm
import Levenshtein
from tqdm import tqdm

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_ctc.pth"

# =====================================================================
# 2. DATASET (Modified for CTC Length Alignments)
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, processor,
                 transform=None, max_target_length=25):
        self.data_dir          = data_dir
        self.transform         = transform
        self.processor         = processor
        self.max_target_length = max_target_length
        self.samples           = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            print(f"Error: {path} not found.")
            return
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9:
                    continue
                word_id       = parts[0]
                status        = parts[1]
                if status != "ok":
                    continue
                transcription = " ".join(parts[8:])
                if not transcription.strip():
                    continue
                id_parts           = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent      = f"{id_parts[0]}-{id_parts[1]}"
                filename           = f"{word_id}.png"
                img_path = os.path.join(
                    self.data_dir, "words",
                    folder_grandparent, folder_parent, filename
                )
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription))
        print(f"Registered {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        attempts = 0
        while attempts < len(self.samples):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
                break
            except Exception:
                attempts += 1
                idx = random.randint(0, len(self.samples) - 1)
        if attempts >= len(self.samples):
            raise RuntimeError("All images appear corrupted.")
        if self.transform:
            image = self.transform(image)
        
        # CTC encoding does not use standard target padding masks like -100
        labels = self.processor(
            text,
            padding=False,
            truncation=True,
            max_length=self.max_target_length,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        return image, labels, text


def get_iam_dataloaders(data_dir, words_txt_path, processor, batch_size=32):
    train_transform = T.Compose([
        T.Resize((64, 256)),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.3),
        T.RandomRotation(degrees=3),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((64, 256)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, processor=processor, transform=None)
    total      = len(full_dataset)
    train_size = int(0.9 * total)

    generator     = torch.Generator().manual_seed(42)
    indices       = torch.randperm(total, generator=generator).tolist()
    train_indices = indices[:train_size]
    val_indices   = indices[train_size:]

    train_samples = [full_dataset.samples[i] for i in train_indices]
    val_samples   = [full_dataset.samples[i] for i in val_indices]

    class SplitDataset(Dataset):
        def __init__(self, samples, processor, transform, max_target_length=25):
            self.samples           = samples
            self.processor         = processor
            self.transform         = transform
            self.max_target_length = max_target_length

        def __len__(self):
            return len(self.samples)

        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            try:
                image = Image.open(img_path).convert('RGB')
            except Exception:
                image = Image.new('RGB', (256, 64), color=255)
            if self.transform:
                image = self.transform(image)
            
            labels = self.processor(
                text,
                padding=False,
                truncation=True,
                max_length=self.max_target_length,
                return_tensors="pt"
            ).input_ids.squeeze(0)
            
            return image, labels, text

    # CTC needs custom collation to handle variable target lengths smoothly
    def ctc_collate_fn(batch):
        images, labels, texts = zip(*batch)
        images = torch.stack(images, dim=0)
        target_lengths = torch.tensor([len(lbl) for lbl in labels], dtype=torch.long)
        targets = torch.cat(labels) # Flatten labels into 1D array for CTCLoss
        return images, targets, target_lengths, texts

    train_dataset = SplitDataset(train_samples, processor, train_transform)
    val_dataset   = SplitDataset(val_samples, processor, val_transform)
    
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                               num_workers=4, drop_last=True, collate_fn=ctc_collate_fn)
    val_loader    = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                               num_workers=4, collate_fn=ctc_collate_fn)
    return train_loader, val_loader


# =====================================================================
# 3. ARCHITECTURE (Swin + BiLSTM + CTC Head)
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B    = x.size(0)
        pts  = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B              = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:, :, 0].mean(dim=1)
        cy = control_points[:, :, 1].mean(dim=1)
        theta          = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat       = feat.view(B, H*W, C)  # (B, ~64, 512)
        return self.proj(feat)             # (B, ~64, 768)


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn      = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm       = nn.LSTM(
            input_size    = d_model,
            hidden_size   = d_model // 2,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = 0.3
        )
        
        # RoBERTa is replaced completely by this direct linear token classifier
        print(f"Initializing Linear CTC Token Classifier Head (Size: {vocab_size})...")
        self.ctc_head = nn.Linear(d_model, vocab_size)

    def forward(self, images):
        rectified = self.tps_stn(images)          
        swin_feat = self.swin_encoder(rectified)  
        memory, _ = self.bilstm(swin_feat)         
        
        # Logits output shape: [Batch, Sequence_Length, Vocab_Size]
        logits = self.ctc_head(memory)
        return logits

    @torch.no_grad()
    def generate(self, images):
        """
        Replaces the slow autoregressive for-loop generation with instantaneous parallel CTC decoding.
        """
        logits = self.forward(images) # [B, T, Vocab]
        preds = logits.argmax(dim=-1) # [B, T]
        return preds


# =====================================================================
# 4. AGENTIC VERIFICATION MODULE
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = set()
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        self.lexicon.add(word)
        print(f"Lexicon built: {len(self.lexicon)} words.")

    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
        best_candidate = text_output
        min_distance   = float('inf')
        target_len     = len(cleaned)
        for word in self.lexicon:
            if abs(len(word) - target_len) > 1:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist < min_distance and dist == 1:
                min_distance   = dist
                best_candidate = word
        if min_distance == 1:
            return (best_candidate.upper() if text_output.isupper() else best_candidate)
        return text_output


# =====================================================================
# 5. METRICS & EARLY STOPPING
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        pred_c   = pred.strip()
        target_c = target.strip()
        dist     = Levenshtein.distance(pred_c, target_c)
        if len(target_c) > 0:
            total_cer += dist / len(target_c)
        elif len(pred_c) > 0:
            total_cer += 1.0
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_metric = float('inf')
        self.early_stop = False

    def __call__(self, current_metric):
        if current_metric < self.best_metric - self.min_delta:
            self.best_metric = current_metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


def ctc_greedy_decode(tokens, tokenizer):
    """
    Standard CTC Decoding: collapses consecutive duplicates and drops pad/blank tokens.
    """
    decoded_strings = []
    blank_id = tokenizer.pad_token_id # Using standard padding index as the CTC Blank token
    
    for seq in tokens:
        collapsed = []
        prev_token = None
        for token_id in seq.tolist():
            if token_id != prev_token:
                if token_id != blank_id and token_id != tokenizer.bos_token_id and token_id != tokenizer.eos_token_id:
                    collapsed.append(token_id)
                prev_token = token_id
        decoded_strings.append(tokenizer.decode(collapsed, skip_special_tokens=True))
    return decoded_strings


# =====================================================================
# 6. TRAINING PIPELINE
# =====================================================================
def run_training_pipeline(epochs=50, batch_size=32, lr=5e-5, early_stopping_patience=4):
    print("Loading tokenizer mapping configurations...")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

    train_loader, val_loader = get_iam_dataloaders(
        BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_params   = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total model parameters: {total_params/1e6:.2f}M (Significantly reduced without RoBERTa)")
    print(f"Trainable parameters  : {trainable_params/1e6:.2f}M")

    # Switched to CTCLoss
    criterion = nn.CTCLoss(blank=tokenizer.pad_token_id, zero_infinity=True)

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6, verbose=True)
    early_stopper = EarlyStopping(patience=early_stopping_patience, min_delta=0.0)
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)
    
    best_val_wer = float('inf')
    swin_unfrozen = False

    print(f"\nTraining Model on: {device}")

    for epoch in range(1, epochs + 1):
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(), 'lr': 1e-6},
                {'params': model.swin_encoder.proj.parameters(), 'lr': lr},
                {'params': model.bilstm.parameters(), 'lr': lr},
                {'params': model.ctc_head.parameters(), 'lr': lr},
                {'params': model.tps_stn.parameters(), 'lr': lr},
            ], weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6, verbose=True)
            swin_unfrozen = True
            print("Swin encoder unfrozen.")

        # ── Train Step ──────────────────────────────────────────
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        
        for images, targets, target_lengths, _ in pbar:
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            
            # Predict sequence matrix
            logits = model(images) # [B, T, Vocab]
            
            # Formulate inputs for PyTorch CTCLoss: (Time, Batch, Vocab)
            log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)
            input_lengths = torch.full((images.size(0),), log_probs.size(0), dtype=torch.long, device=device)
            
            loss = criterion(log_probs, targets, input_lengths, target_lengths)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} | Train Loss (CTC): {avg_loss:.4f}")

        # ── Validation Step ───────────────────────────────────────
        model.eval()
        all_preds, all_targets = [], []
        first_batch_done = False

        with torch.no_grad():
            for images, _, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                pred_tokens = model.generate(images)
                
                # Dynamic parallel string decoding via CTC Collapse algorithm
                raw_decoded = ctc_greedy_decode(pred_tokens, tokenizer)

                if not first_batch_done:
                    print(f"\n--- Debug Decoding Matrix Epoch {epoch} ---")
                    for i in range(min(3, images.size(0))):
                        print(f"Target: '{text_labels[i]}' | Native CTC Pred: '{raw_decoded[i].strip()}'")
                    print("-------------------------------------------\n")
                    first_batch_done = True

                for i in range(images.size(0)):
                    raw_str = raw_decoded[i]
                    # Agent edits the spelling mistakes produced by the parallel classification head
                    verified_str = agent_verifier.verify_and_correct(raw_str)
                    
                    all_preds.append(verified_str)
                    all_targets.append(text_labels[i])

        metrics = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        scheduler.step(current_wer)
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_val_wer,
                'cer': metrics['CER']
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved | Best Val WER: {best_val_wer:.2f}%")
        
        if early_stopper(current_wer):
            print(f"\nEarly stopping triggered! Training complete at epoch {epoch}.")
            break
            
        print("=" * 70)


if __name__ == "__main__":
    run_training_pipeline(epochs=50, batch_size=32, lr=5e-5, early_stopping_patience=4)

Loading tokenizer mapping configurations...


/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Registered 38305 samples.


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Initializing Linear CTC Token Classifier Head (Size: 50265)...


...

Epoch 50 | Train Loss (CTC): 0.4279


Epoch 50/50 [Val]:   1%|▍                                                 | 1/120 [00:00<00:21,  5.48it/s]


--- Debug Decoding Matrix Epoch 50 ---
Target: 'any' | Native CTC Pred: 'any'
Target: 'any' | Native CTC Pred: 'any'
Target: 'unopposed' | Native CTC Pred: 'reg'
-------------------------------------------



Epoch 50/50 [Val]: 100%|████████████████████████████████████████████████| 120/120 [00:07<00:00, 16.38it/s]


Epoch 50 | CER: 25.93% | WER: 36.88%
EarlyStopping counter: 1 out of 4


In [6]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T
from transformers import RobertaTokenizer
from tqdm import tqdm
import timm

# ============================================================================
# 1. Model definitions (EXACTLY as in training – no attention, img_size=(64,256))
# ============================================================================

class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:, :, 0].mean(dim=1)
        cy = control_points[:, :, 1].mean(dim=1)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(64, 256),          # MUST match training
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.proj(feat)


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        self.ctc_head = nn.Linear(d_model, vocab_size)

    def forward(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        logits = self.ctc_head(memory)
        return logits

    @torch.no_grad()
    def generate(self, images):
        logits = self.forward(images)
        preds = logits.argmax(dim=-1)
        return preds


# ============================================================================
# 2. Paths and setup
# ============================================================================
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_ctc.pth"
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================================
# 3. Load tokenizer and model
# ============================================================================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

model = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

# Load checkpoint with strict=False to ignore missing/unexpected keys
# (in case of minor differences, but we expect exact match now)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} | Best WER: {checkpoint['best_wer']:.2f}%")

# ============================================================================
# 4. Preprocessing (must match validation transform from training)
# ============================================================================
val_transform = T.Compose([
    T.Resize((64, 256)),              # Same as training
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    tensor = val_transform(image).unsqueeze(0)  # [1,3,64,256]
    return tensor

# ============================================================================
# 5. CTC greedy decoding (same as training)
# ============================================================================
def ctc_greedy_decode(tokens, tokenizer):
    decoded_strings = []
    blank_id = tokenizer.pad_token_id
    for seq in tokens:
        collapsed = []
        prev_token = None
        for token_id in seq.tolist():
            if token_id != prev_token:
                if token_id != blank_id and token_id != tokenizer.bos_token_id and token_id != tokenizer.eos_token_id:
                    collapsed.append(token_id)
                prev_token = token_id
        decoded_strings.append(tokenizer.decode(collapsed, skip_special_tokens=True))
    return decoded_strings

# ============================================================================
# 6. Single image prediction
# ============================================================================
def predict_single_image(image_path):
    input_tensor = preprocess_image(image_path).to(device)
    with torch.no_grad():
        logits = model(input_tensor)
        pred_tokens = logits.argmax(dim=-1)
        decoded = ctc_greedy_decode(pred_tokens, tokenizer)
    return decoded[0]

# Example: print(predict_single_image("path/to/word.png"))

# ============================================================================
# 7. (Optional) Evaluate on validation set – reuse training split logic
# ============================================================================
def get_validation_loader(batch_size=32):
    # You need to have IAMWordsDataset defined (same as training)
    # For brevity, here's a simplified version that recreates the exact val split
    from torch.utils.data import Dataset, DataLoader

    class ValDataset(Dataset):
        def __init__(self, samples, processor, transform):
            self.samples = samples
            self.processor = processor
            self.transform = transform
        def __len__(self):
            return len(self.samples)
        def __getitem__(self, idx):
            img_path, text = self.samples[idx]
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            labels = self.processor(text, padding=False, truncation=True, max_length=25, return_tensors="pt").input_ids.squeeze(0)
            return image, labels, text

    # Build full dataset (without transform) to get samples
    from torch.utils.data import Dataset as BaseDataset
    class TempDataset(BaseDataset):
        def __init__(self, data_dir, words_txt_path):
            self.samples = []
            # parse words.txt as in training
            with open(words_txt_path, 'r') as f:
                for line in f:
                    if line.startswith('#') or not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) < 9 or parts[1] != "ok": continue
                    transcription = " ".join(parts[8:]).strip()
                    if not transcription: continue
                    word_id = parts[0]
                    id_parts = word_id.split('-')
                    folder_grandparent = id_parts[0]
                    folder_parent = f"{id_parts[0]}-{id_parts[1]}"
                    filename = f"{word_id}.png"
                    img_path = os.path.join(data_dir, "words", folder_grandparent, folder_parent, filename)
                    if os.path.exists(img_path):
                        self.samples.append((img_path, transcription))
        def __len__(self): return len(self.samples)
        def __getitem__(self, idx): return self.samples[idx]

    full = TempDataset(BASE_DATA_DIR, WORDS_TXT_FILE)
    total = len(full)
    train_size = int(0.9 * total)
    generator = torch.Generator().manual_seed(42)
    indices = torch.randperm(total, generator=generator).tolist()
    val_indices = indices[train_size:]
    val_samples = [full.samples[i] for i in val_indices]

    val_dataset = ValDataset(val_samples, tokenizer, val_transform)
    def collate_fn(batch):
        images, labels, texts = zip(*batch)
        images = torch.stack(images, dim=0)
        target_lengths = torch.tensor([len(lbl) for lbl in labels], dtype=torch.long)
        targets = torch.cat(labels)
        return images, targets, target_lengths, texts
    return DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, collate_fn=collate_fn)

def calculate_metrics(predictions, targets):
    import Levenshtein
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        dist = Levenshtein.distance(pred.strip(), target.strip())
        total_cer += dist / max(len(target), 1)
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {"CER": round((total_cer / n) * 100, 2), "WER": round((wrong_words / n) * 100, 2)}

def evaluate_model(val_loader):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, _, _, text_labels in tqdm(val_loader, desc="Evaluating"):
            images = images.to(device)
            logits = model(images)
            pred_tokens = logits.argmax(dim=-1)
            raw_decoded = ctc_greedy_decode(pred_tokens, tokenizer)
            for i in range(len(raw_decoded)):
                all_preds.append(raw_decoded[i].strip())
                all_targets.append(text_labels[i])
    metrics = calculate_metrics(all_preds, all_targets)
    print(f"Validation CER: {metrics['CER']:.2f}% | WER: {metrics['WER']:.2f}%")
    return metrics

# Uncomment to run evaluation:
val_loader = get_validation_loader(batch_size=32)
evaluate_model(val_loader)

# ============================================================================
# 8. Test on a single image (example)
# ============================================================================
# result = predict_single_image("/path/to/your/word_image.png")
# print("Predicted text:", result)

Using device: cuda


/tmp/ipykernel_3417077/51963886.py:127: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)


Loaded checkpoint from epoch 49 | Best WER: 36.67%


Evaluating: 100%|███████████████████████████████████████████████████████| 120/120 [00:06<00:00, 17.72it/s]

Validation CER: 25.34% | WER: 37.01%


{'CER': 25.34, 'WER': 37.01}

## 2

In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm
import Levenshtein
from tqdm import tqdm
import numpy as np

# =====================================================================
# 1. PATHS
# =====================================================================
BASE_DATA_DIR   = "/home/mca24-26/handwritten text detection/dataset/iam/iam_words"
WORDS_TXT_FILE  = os.path.join(BASE_DATA_DIR, "words.txt")
CHECKPOINT_PATH = "/home/mca24-26/handwritten text detection/iam_htr_ctc.pth"

# =====================================================================
# 2. CHARACTER TOKENIZER
# =====================================================================
def build_char_set_from_words_txt(words_txt_path):
    """Extract all characters appearing in transcriptions."""
    chars = set()
    with open(words_txt_path, 'r') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) < 9 or parts[1] != "ok":
                continue
            text = " ".join(parts[8:]).strip().lower()
            chars.update(text)
    chars = sorted(list(chars))
    print(f"Character set size: {len(chars)}")
    return chars

class CharTokenizer:
    """CTC-friendly character tokenizer. Blank token = 0."""
    def __init__(self, chars):
        self.chars = chars
        self.char2idx = {ch: i+1 for i, ch in enumerate(chars)}  # 0 reserved for blank
        self.idx2char = {i+1: ch for ch, i in self.char2idx.items()}
        self.blank_id = 0
        self.vocab_size = len(chars) + 1

    def encode(self, text, max_length=None):
        text = text.lower()
        ids = [self.char2idx[ch] for ch in text if ch in self.char2idx]
        if max_length is not None and len(ids) > max_length:
            ids = ids[:max_length]
        return torch.tensor(ids, dtype=torch.long)

    def decode(self, ids):
        """Standard fallback decoder."""
        return ''.join([self.idx2char[idx] for idx in ids if idx in self.idx2char])

# =====================================================================
# 3. ELASTIC DEFORMATION (common for HTR)
# =====================================================================
class ElasticTransform:
    def __init__(self, alpha=50, sigma=5):
        self.alpha = alpha
        self.sigma = sigma

    def __call__(self, img):
        if random.random() < 0.5:
            return img
        img_np = np.array(img)
        shape = img_np.shape[:2]
        dx = np.random.randn(shape[0], shape[1]) * self.sigma
        dy = np.random.randn(shape[0], shape[1]) * self.sigma
        dx = cv2.GaussianBlur(dx, (0,0), self.sigma) * self.alpha
        dy = cv2.GaussianBlur(dy, (0,0), self.sigma) * self.alpha
        x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
        map_x = (x + dx).astype(np.float32)
        map_y = (y + dy).astype(np.float32)
        distorted = cv2.remap(img_np, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        return Image.fromarray(distorted)

try:
    import cv2
    ELASTIC_AVAILABLE = True
except ImportError:
    ELASTIC_AVAILABLE = False
    print("Warning: OpenCV not installed. Elastic deformation disabled.")

# =====================================================================
# 4. DATASET WITH DYNAMIC PADDING
# =====================================================================
class IAMWordsDataset(Dataset):
    def __init__(self, data_dir, words_txt_path, tokenizer, transform=None, max_width=512, max_height=128):
        self.data_dir = data_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_width = max_width
        self.max_height = max_height
        self.samples = []
        self._parse_words_txt(words_txt_path)

    def _parse_words_txt(self, path):
        if not os.path.exists(path):
            raise FileNotFoundError(f"{path} not found.")
        with open(path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) < 9 or parts[1] != "ok":
                    continue
                transcription = " ".join(parts[8:]).strip()
                if not transcription:
                    continue
                word_id = parts[0]
                id_parts = word_id.split('-')
                folder_grandparent = id_parts[0]
                folder_parent = f"{id_parts[0]}-{id_parts[1]}"
                filename = f"{word_id}.png"
                img_path = os.path.join(self.data_dir, "words", folder_grandparent, folder_parent, filename)
                if os.path.exists(img_path):
                    self.samples.append((img_path, transcription.lower()))
        print(f"Loaded {len(self.samples)} samples.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (100, 32), color=255)
        if self.transform:
            image = self.transform(image)
        labels = self.tokenizer.encode(text)
        return image, labels, text

def collate_fn_ctc(batch):
    images, labels, texts = zip(*batch)
    max_h = max(img.shape[1] for img in images)
    max_w = max(img.shape[2] for img in images)
    padded_images = []
    for img in images:
        _, h, w = img.shape
        pad_h = max_h - h
        pad_w = max_w - w
        padded = F.pad(img, (0, pad_w, 0, pad_h), value=0)
        padded_images.append(padded)
    images = torch.stack(padded_images, dim=0)

    target_lengths = torch.tensor([len(lbl) for lbl in labels], dtype=torch.long)
    targets = torch.cat(labels)
    return images, targets, target_lengths, texts

def get_iam_dataloaders(data_dir, words_txt_path, tokenizer, batch_size=32):
    train_transform = T.Compose([
        T.Resize((128, 512)),
        T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5),
        T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.2),
        T.RandomRotation(degrees=2),
        ElasticTransform() if ELASTIC_AVAILABLE else lambda x: x,
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    val_transform = T.Compose([
        T.Resize((128, 512)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    full_dataset = IAMWordsDataset(data_dir, words_txt_path, tokenizer, transform=None)
    total = len(full_dataset)
    train_size = int(0.9 * total)
    indices = torch.randperm(total).tolist()
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]

    train_dataset = IAMWordsDataset(data_dir, words_txt_path, tokenizer, transform=train_transform)
    val_dataset = IAMWordsDataset(data_dir, words_txt_path, tokenizer, transform=val_transform)

    train_dataset.samples = [train_dataset.samples[i] for i in train_indices]
    val_dataset.samples = [val_dataset.samples[i] for i in val_indices]

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, collate_fn=collate_fn_ctc, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, collate_fn=collate_fn_ctc)
    return train_loader, val_loader

# =====================================================================
# 5. MODEL ARCHITECTURE
# =====================================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4,4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Linear(256, num_control_points*2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B = x.size(0)
        control_points = self.localization(x)
        cx = control_points[:,:,0].mean(dim=1)
        cy = control_points[:,:,1].mean(dim=1)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:,0,0] = 1.0
        theta[:,1,1] = 1.0
        theta[:,0,2] = torch.tanh(cx) * 0.05
        theta[:,1,2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model("swin_base_patch4_window7_224", pretrained=True, features_only=True, img_size=(128,512), strict_img_size=False)
        self.proj = nn.Linear(512, d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]  # stage 3 output [B, H, W, 512]
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.proj(feat)  # [B, T, d_model]

class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16, use_attention=True):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(input_size=d_model, hidden_size=d_model//2, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.use_attention = use_attention
        if use_attention:
            self.attention = nn.MultiheadAttention(d_model, num_heads=4, dropout=0.1, batch_first=True)
            self.attn_norm = nn.LayerNorm(d_model)
        self.ctc_head = nn.Linear(d_model, vocab_size)

    def forward(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)  # [B, T, d_model]
        memory, _ = self.bilstm(swin_feat)         # [B, T, d_model]
        if self.use_attention:
            attn_out, _ = self.attention(memory, memory, memory)
            memory = self.attn_norm(memory + attn_out)
        logits = self.ctc_head(memory)             # [B, T, vocab_size]
        return logits

    @torch.no_grad()
    def generate(self, images):
        """Performs fast GPU-accelerated greedy decoding across the batch."""
        logits = self.forward(images)             # [B, T, V]
        return ctc_fast_greedy_decode(logits, blank_id=0)

# =====================================================================
# 6. FAST GREEDY DECODING (Vectorized across batch)
# =====================================================================
def ctc_fast_greedy_decode(logits, blank_id=0):
    """
    Highly optimized batch-wise argmax/greedy deduction for CTC.
    logits: tensor of shape [B, T, V]
    """
    # [B, T] - Get best index per timestep
    best_path = torch.argmax(logits, dim=-1).cpu().numpy()
    
    decoded_strings = []
    for b in range(best_path.shape[0]):
        seq = best_path[b]
        # Drop adjacent duplicates using numpy indexing rules
        changes = seq[1:] != seq[:-1]
        # Always retain the initial character element
        clean_seq = seq[np.concatenate(([True], changes))]
        # Strip out the padding/blank tokens
        final_tokens = [tok for tok in clean_seq if tok != blank_id]
        
        # Translate matrix indexes to characters using the global tokenizer map
        decoded_str = "".join([tokenizer.idx2char[idx] for idx in final_tokens if idx in tokenizer.idx2char])
        decoded_strings.append(decoded_str)
        
    return decoded_strings

# =====================================================================
# 7. AGENTIC VERIFICATION (Levenshtein Lexicon Correction)
# =====================================================================
class AgenticVerificationModule:
    def __init__(self, words_txt_file=None):
        self.lexicon = set()
        if words_txt_file and os.path.exists(words_txt_file):
            self._build_lexicon(words_txt_file)

    def _build_lexicon(self, file_path):
        with open(file_path, 'r') as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.strip().split()
                if len(parts) >= 9 and parts[1] == "ok":
                    word = " ".join(parts[8:]).strip().lower()
                    if word:
                        self.lexicon.add(word)
        print(f"Lexicon built: {len(self.lexicon)} words.")

    def verify_and_correct(self, text_output):
        cleaned = text_output.strip().lower()
        if not cleaned or cleaned in self.lexicon or len(cleaned) <= 2 or any(c.isdigit() for c in cleaned):
            return text_output
        best_candidate = text_output
        min_distance = float('inf')
        target_len = len(cleaned)
        for word in self.lexicon:
            if abs(len(word) - target_len) > 2:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist < min_distance and dist <= 2:
                min_distance = dist
                best_candidate = word
        if min_distance <= 2:
            return best_candidate
        return text_output

# =====================================================================
# 8. METRICS & TRAINING LOOP
# =====================================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        pred_c = pred.strip()
        target_c = target.strip()
        dist = Levenshtein.distance(pred_c, target_c)
        if len(target_c) > 0:
            total_cer += dist / len(target_c)
        if dist != 0:
            wrong_words += 1
    n = len(targets)
    return {"CER": round((total_cer / n) * 100, 4), "WER": round((wrong_words / n) * 100, 4)}

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_metric = float('inf')
        self.early_stop = False

    def __call__(self, current_metric):
        if current_metric < self.best_metric - self.min_delta:
            self.best_metric = current_metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

def run_training_pipeline(epochs=50, batch_size=24, lr=5e-5, early_stopping_patience=4, use_attention=True):
    chars = build_char_set_from_words_txt(WORDS_TXT_FILE)
    global tokenizer
    tokenizer = CharTokenizer(chars)
    vocab_size = tokenizer.vocab_size
    print(f"Vocabulary size: {vocab_size} (including blank)")

    train_loader, val_loader = get_iam_dataloaders(BASE_DATA_DIR, WORDS_TXT_FILE, tokenizer, batch_size=batch_size)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CompleteHTRPipeline(vocab_size=vocab_size, use_attention=use_attention).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params/1e6:.2f}M | Trainable: {trainable_params/1e6:.2f}M")

    criterion = nn.CTCLoss(blank=tokenizer.blank_id, zero_infinity=True)

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False
    print("Swin frozen for first 3 epochs.")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6, verbose=True)
    early_stopper = EarlyStopping(patience=early_stopping_patience, min_delta=0.0)
    agent_verifier = AgenticVerificationModule(words_txt_file=WORDS_TXT_FILE)

    best_val_wer = float('inf')
    swin_unfrozen = False

    for epoch in range(1, epochs+1):
        if epoch == 4 and not swin_unfrozen:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.swin_encoder.swin.parameters(), 'lr': 1e-6},
                {'params': model.swin_encoder.proj.parameters(), 'lr': lr},
                {'params': model.bilstm.parameters(), 'lr': lr},
                {'params': model.ctc_head.parameters(), 'lr': lr},
                {'params': model.tps_stn.parameters(), 'lr': lr},
            ], weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6, verbose=True)
            swin_unfrozen = True
            print("Swin encoder unfrozen.")

        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")
        for images, targets, target_lengths, _ in pbar:
            images = images.to(device)
            targets = targets.to(device)
            optimizer.zero_grad()
            logits = model(images)  
            log_probs = F.log_softmax(logits, dim=-1).permute(1, 0, 2)  # [T, B, V]
            input_lengths = torch.full((images.size(0),), log_probs.size(0), dtype=torch.long, device=device)
            loss = criterion(log_probs, targets, input_lengths, target_lengths)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f}")

        # Validation
        model.eval()
        all_preds, all_targets = [], []
        first_batch = True
        with torch.no_grad():
            for images, _, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                # Fast greedy decoding
                pred_texts = model.generate(images)
                if first_batch:
                    print("\n--- Sample predictions ---")
                    for i in range(min(3, len(pred_texts))):
                        print(f"Target: '{text_labels[i]}' | Pred: '{pred_texts[i]}'")
                    print("--------------------------\n")
                    first_batch = False
                for i in range(len(pred_texts)):
                    verified = agent_verifier.verify_and_correct(pred_texts[i])
                    all_preds.append(verified)
                    all_targets.append(text_labels[i])
        metrics = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        scheduler.step(current_wer)
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_val_wer,
                'cer': metrics['CER']
            }, CHECKPOINT_PATH)
            print(f"Checkpoint saved (WER: {best_val_wer:.2f}%)")
        if early_stopper(current_wer):
            print(f"Early stopping at epoch {epoch}.")
            break
        print("=" * 70)

if __name__ == "__main__":
    run_training_pipeline(epochs=50, batch_size=24, lr=5e-5, early_stopping_patience=4, use_attention=True)

Character set size: 51
Vocabulary size: 52 (including blank)
Loaded 38305 samples.
Loaded 38305 samples.
Loaded 38305 samples.
Total parameters: 97.25M | Trainable: 97.25M
Swin frozen for first 3 epochs.
Lexicon built: 6231 words.


Epoch 1/50 [Train]: 100%|████████████████████████████████| 1436/1436 [15:42<00:00,  1.52it/s, loss=2.7401]


Epoch 1 | Train Loss: 3.3667


Epoch 1/50 [Val]:   1%|▎                                                  | 1/160 [00:00<01:13,  2.16it/s]


--- Sample predictions ---
Target: 'prefer' | Pred: ''
Target: 'it' | Pred: ''
Target: 'innocent' | Pred: ''
--------------------------



...

Epoch 11 | Train Loss: 1.2537


Epoch 11/50 [Val]:   1%|▎                                                 | 1/160 [00:00<01:24,  1.89it/s]


--- Sample predictions ---
Target: 'prefer' | Pred: 'odsdq'
Target: 'it' | Pred: 'es'
Target: 'innocent' | Pred: 'hmlndc'
--------------------------



Epoch 11/50 [Val]: 100%|████████████████████████████████████████████████| 160/160 [00:47<00:00,  3.36it/s]

Epoch 11 | CER: 97.17% | WER: 99.95%
Early stopping at epoch 11.
